# Modeling — IHSG Multivariate LSTM
## Analisis Prediksi IHSG Menggunakan Multivariate LSTM Berdasarkan Data Historis dan Variabel Makroekonomi Periode 2015–2024

**Alur notebook ini:**
1. Load artefak preprocessing (`sequences_dual`, `sequences_uni`, scalers, metadata)
2. Definisi arsitektur: `build_lstm_dual_branch()` & `build_lstm_univariate()`
3. **Grid Search** — eksperimen semua kombinasi `seq_daily × seq_monthly × batch_size × lstm_units`
   dengan GRID_HORIZON epoch (proxy cepat) → pilih best config per model
4. **Final Training** — train ulang best config dengan epoch penuh + EarlyStopping
5. Evaluasi pada test set: RMSE, MAPE, R²
6. **Simpan artefak model** ke `model/` — siap dipanggil kapan saja untuk forecast/visualisasi

**Model yang dilatih:**
| Model | Input | Arsitektur |
|---|---|---|
| Univariate LSTM | IHSG only (daily) | Sequential LSTM |
| Multivariate LSTM | IHSG + Kurs (daily) + Inflasi + BI Rate (monthly) | Dual-Branch Functional API |


## 1. Imports & Environment

In [1]:
import os
import gc
import json
import warnings
import itertools
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from datetime import datetime

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')
np.random.seed(42)

import tensorflow as tf
tf.random.set_seed(42)

from tensorflow.keras import Input, Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Concatenate, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import mean_squared_error, r2_score

print(f"TensorFlow  : {tf.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"Pandas      : {pd.__version__}")
print(f"GPU devices : {tf.config.list_physical_devices('GPU') or 'None (CPU only)'}")


TensorFlow  : 2.20.0
NumPy       : 2.2.6
Pandas      : 2.3.3
GPU devices : None (CPU only)


## 2. Path & Direktori

In [9]:
ROOT           = Path(os.getcwd()).parent
DATA_PROCESSED = ROOT / "data" / "processed"
MODEL_DIR = ROOT / "model" / "model_multivariate"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT          :", ROOT)
print("DATA_PROCESSED:", DATA_PROCESSED)
print("MODEL_DIR     :", MODEL_DIR)


ROOT          : d:\project_skripsi
DATA_PROCESSED: d:\project_skripsi\data\processed
MODEL_DIR     : d:\project_skripsi\model\model_multivariate


## 3. Load Artefak Preprocessing

In [10]:
# ── Metadata ──
with open(DATA_PROCESSED / "preprocessing_meta.json") as f:
    meta = json.load(f)

GRID_CONFIG      = meta['grid_config']
DAILY_FEATURES   = meta['daily_features']
MONTHLY_FEATURES = meta['monthly_features']
TARGET_COL       = meta['target_col']
TARGET_IDX       = meta['target_idx_in_daily']

print("=== Metadata Preprocessing ===")
print(f"  Daily  train/test : {meta['n_daily_train']} / {meta['n_daily_test']} hari")
print(f"  Monthly train/test: {meta['n_monthly_train']} / {meta['n_monthly_test']} bulan")
print(f"  Cutoff date       : {meta['cutoff_date']}")
print(f"  Daily features    : {DAILY_FEATURES}")
print(f"  Monthly features  : {MONTHLY_FEATURES}")
print(f"  Target            : {TARGET_COL}")
print(f"  Grid config       : {GRID_CONFIG}")
print(f"  Kombinasi dual    : {meta['available_dual_keys']}")


=== Metadata Preprocessing ===
  Daily  train/test : 1940 / 486 hari
  Monthly train/test: 95 / 25 bulan
  Cutoff date       : 2022-12-21
  Daily features    : ['ihsg', 'nilai_tukar']
  Monthly features  : ['inflasi', 'bi_rate']
  Target            : ihsg
  Grid config       : {'seq_daily_candidates': [22, 60, 90], 'seq_monthly_candidates': [3, 6, 12], 'batch_size_candidates': [16, 32], 'grid_horizon': 5, 'test_ratio': 0.2, 'random_seed': 42}
  Kombinasi dual    : ['(22, 3)', '(22, 6)', '(22, 12)', '(60, 3)', '(60, 6)', '(60, 12)', '(90, 3)', '(90, 6)', '(90, 12)']


In [11]:
# ── Scalers ──
scaler_daily   = joblib.load(DATA_PROCESSED / "scaler_daily.joblib")
scaler_monthly = joblib.load(DATA_PROCESSED / "scaler_monthly.joblib")
scaler_target  = joblib.load(DATA_PROCESSED / "scaler_target.joblib")
scaler_uni     = joblib.load(DATA_PROCESSED / "scaler_univariate.joblib")

print(f"✓ scaler_daily    : {scaler_daily.n_features_in_} features")
print(f"✓ scaler_monthly  : {scaler_monthly.n_features_in_} features")
print(f"✓ scaler_target   : {scaler_target.n_features_in_} feature (IHSG)")
print(f"✓ scaler_uni      : {scaler_uni.n_features_in_} feature")


✓ scaler_daily    : 2 features
✓ scaler_monthly  : 2 features
✓ scaler_target   : 1 feature (IHSG)
✓ scaler_uni      : 1 feature


In [12]:
# ── Sequences Dual-Branch ──
print("Loading sequences_dual.joblib ...", end=" ", flush=True)
sequences_dual = joblib.load(DATA_PROCESSED / "sequences_dual.joblib")
print(f"OK — {len(sequences_dual)} kombinasi")

for key, v in sequences_dual.items():
    print(f"   seq_daily={key[0]:3d}, seq_monthly={key[1]:2d}  "
          f"Train: X_d={v['X_daily_train'].shape}, X_m={v['X_monthly_train'].shape}, y={v['y_train'].shape}  "
          f"| Test: X_d={v['X_daily_test'].shape}, X_m={v['X_monthly_test'].shape}, y={v['y_test'].shape}")

print()

# ── Sequences Univariate ──
print("Loading sequences_uni.joblib ...", end=" ", flush=True)
sequences_uni = joblib.load(DATA_PROCESSED / "sequences_uni.joblib")
print(f"OK — {len(sequences_uni)} seq_daily kandidat")
for k, v in sequences_uni.items():
    print(f"   seq_daily={k}: X_train={v['X_train'].shape}, y_train={v['y_train'].shape} "
          f"| X_test={v['X_test'].shape}, y_test={v['y_test'].shape}")


Loading sequences_dual.joblib ... OK — 9 kombinasi
   seq_daily= 22, seq_monthly= 3  Train: X_d=(1864, 22, 2), X_m=(1864, 3, 2), y=(1864,)  | Test: X_d=(485, 22, 2), X_m=(485, 3, 2), y=(485,)
   seq_daily= 22, seq_monthly= 6  Train: X_d=(1803, 22, 2), X_m=(1803, 6, 2), y=(1803,)  | Test: X_d=(485, 22, 2), X_m=(485, 6, 2), y=(485,)
   seq_daily= 22, seq_monthly=12  Train: X_d=(1682, 22, 2), X_m=(1682, 12, 2), y=(1682,)  | Test: X_d=(485, 22, 2), X_m=(485, 12, 2), y=(485,)
   seq_daily= 60, seq_monthly= 3  Train: X_d=(1864, 60, 2), X_m=(1864, 3, 2), y=(1864,)  | Test: X_d=(485, 60, 2), X_m=(485, 3, 2), y=(485,)
   seq_daily= 60, seq_monthly= 6  Train: X_d=(1803, 60, 2), X_m=(1803, 6, 2), y=(1803,)  | Test: X_d=(485, 60, 2), X_m=(485, 6, 2), y=(485,)
   seq_daily= 60, seq_monthly=12  Train: X_d=(1682, 60, 2), X_m=(1682, 12, 2), y=(1682,)  | Test: X_d=(485, 60, 2), X_m=(485, 12, 2), y=(485,)
   seq_daily= 90, seq_monthly= 3  Train: X_d=(1836, 90, 2), X_m=(1836, 3, 2), y=(1836,)  | Test: X_

## 4. Hyperparameter Grid Search Config

Grid search dilakukan secara **empiris**: setiap kombinasi benar-benar dilatih selama `GRID_HORIZON` epoch (sebagai *proxy* cepat), lalu dipilih konfigurasi dengan validation RMSE terkecil.

Kandidat yang diuji:

| Parameter | Kandidat | Justifikasi |
|---|---|---|
| `seq_daily` | [22, 60, 90] | ~1, 3, 4.5 bulan trading |
| `seq_monthly` | [3, 6, 12] | Triwulan, semester, tahunan |
| `lstm_units` | [64, 128] | Kapasitas model |
| `dropout_rate` | [0.2, 0.3] | Regularisasi |
| `batch_size` | [16, 32] | Dari preprocessing config |
| `learning_rate` | [0.001, 0.0005] | Adam LR |


In [13]:
# ── Grid kandidat ──
GRID = {
    "seq_daily"     : GRID_CONFIG['seq_daily_candidates'],    # [22, 60, 90]
    "seq_monthly"   : GRID_CONFIG['seq_monthly_candidates'],  # [3, 6, 12]
    "lstm_units"    : [64, 128],
    "dropout_rate"  : [0.2, 0.3],
    "batch_size"    : GRID_CONFIG['batch_size_candidates'],   # [16, 32]
    "learning_rate" : [0.001, 0.0005],
}

# Grid search untuk univariate — seq_monthly tidak relevan
GRID_UNI = {
    "seq_daily"     : GRID_CONFIG['seq_daily_candidates'],
    "lstm_units"    : [64, 128],
    "dropout_rate"  : [0.2, 0.3],
    "batch_size"    : GRID_CONFIG['batch_size_candidates'],
    "learning_rate" : [0.001, 0.0005],
}

GRID_HORIZON   = GRID_CONFIG['grid_horizon']   # 5 epoch proxy
FINAL_EPOCHS   = 100                           # epoch final training
PATIENCE_ES    = 15                            # EarlyStopping patience
PATIENCE_LR    = 7                             # ReduceLROnPlateau patience
VAL_SPLIT      = 0.15                          # validation split dari train

# Hitung total kombinasi
n_dual = (len(GRID['seq_daily']) * len(GRID['seq_monthly']) *
          len(GRID['lstm_units']) * len(GRID['dropout_rate']) *
          len(GRID['batch_size']) * len(GRID['learning_rate']))

n_uni  = (len(GRID_UNI['seq_daily']) * len(GRID_UNI['lstm_units']) *
          len(GRID_UNI['dropout_rate']) * len(GRID_UNI['batch_size']) *
          len(GRID_UNI['learning_rate']))

print(f"Multivariate grid: {n_dual} kombinasi")
print(f"Univariate   grid: {n_uni} kombinasi")
print(f"GRID_HORIZON      : {GRID_HORIZON} epoch (proxy)")
print(f"FINAL_EPOCHS      : {FINAL_EPOCHS} epoch (+ EarlyStopping patience={PATIENCE_ES})")
print(f"VAL_SPLIT         : {VAL_SPLIT}")


Multivariate grid: 144 kombinasi
Univariate   grid: 48 kombinasi
GRID_HORIZON      : 5 epoch (proxy)
FINAL_EPOCHS      : 100 epoch (+ EarlyStopping patience=15)
VAL_SPLIT         : 0.15


## 5. Fungsi Build Model

Dua fungsi pembangun arsitektur didefinisikan di sini. `build_lstm_dual_branch()` menyusun model **multivariate** dengan dua cabang LSTM terpisah — satu memproses sekuens harian (IHSG + kurs) dan satu memproses sekuens bulanan (inflasi + BI Rate) — yang kemudian digabung untuk memprediksi IHSG satu langkah ke depan; cabang dipisah karena kedua kelompok fitur memiliki frekuensi dan panjang sekuens yang berbeda. `build_lstm_univariate()` membangun baseline satu cabang yang hanya memakai IHSG sebagai pembanding.

### Arsitektur Dual-Branch (Multivariate)
```
Input_daily (seq_d, 2)          Input_monthly (seq_m, 2)
       │                                │
  LSTM(units, return_seq=True)     LSTM(units, return_seq=True)
  Dropout(rate)                    Dropout(rate)
  LSTM(units//2)                   LSTM(units//2)
  Dropout(rate)                    Dropout(rate)
       │                                │
       └──────── Concatenate() ────────┘
                      │
               Dense(units//4, relu)
               Dropout(rate)
               Dense(1, linear)  ← prediksi IHSG t+1
```

### Arsitektur Univariate (Baseline)
```
Input (seq_d, 1)
LSTM(units, return_seq=True) → Dropout → LSTM(units//2) → Dropout → Dense(1)
```


In [14]:
def build_lstm_dual_branch(seq_daily, seq_monthly,
                            lstm_units=64, dropout_rate=0.2,
                            learning_rate=0.001,
                            n_daily_features=2, n_monthly_features=2):
    """
    Build dual-branch LSTM untuk multivariate forecasting.
    Menggunakan Keras Functional API.
    
    Parameters
    ----------
    seq_daily          : int   — panjang window harian
    seq_monthly        : int   — panjang window bulanan
    lstm_units         : int   — units LSTM layer pertama
    dropout_rate       : float — dropout rate
    learning_rate      : float — Adam learning rate
    n_daily_features   : int   — jumlah fitur harian (default 2: IHSG, Kurs)
    n_monthly_features : int   — jumlah fitur bulanan (default 2: Inflasi, BI Rate)
    
    Returns
    -------
    model : tf.keras.Model  (compiled, siap .fit())
    """
    # ── Branch Daily ──
    inp_daily = Input(shape=(seq_daily, n_daily_features), name='input_daily')
    x_d = LSTM(lstm_units, return_sequences=True, name='lstm_daily_1')(inp_daily)
    x_d = Dropout(dropout_rate, name='drop_daily_1')(x_d)
    x_d = LSTM(lstm_units // 2, return_sequences=False, name='lstm_daily_2')(x_d)
    x_d = Dropout(dropout_rate, name='drop_daily_2')(x_d)
    
    # ── Branch Monthly ──
    inp_monthly = Input(shape=(seq_monthly, n_monthly_features), name='input_monthly')
    x_m = LSTM(lstm_units, return_sequences=True, name='lstm_monthly_1')(inp_monthly)
    x_m = Dropout(dropout_rate, name='drop_monthly_1')(x_m)
    x_m = LSTM(lstm_units // 2, return_sequences=False, name='lstm_monthly_2')(x_m)
    x_m = Dropout(dropout_rate, name='drop_monthly_2')(x_m)
    
    # ── Merge ──
    merged = Concatenate(name='concat')([x_d, x_m])
    x = Dense(lstm_units // 4, activation='relu', name='dense_hidden')(merged)
    x = Dropout(dropout_rate, name='drop_final')(x)
    output = Dense(1, activation='linear', name='output')(x)
    
    model = Model(inputs=[inp_daily, inp_monthly], outputs=output,
                  name='LSTM_DualBranch_Multivariate')
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae']
    )
    return model


def build_lstm_univariate(seq_daily, lstm_units=64, dropout_rate=0.2,
                           learning_rate=0.001):
    """
    Build LSTM univariate (Sequential-style via Functional API).
    Input: IHSG only.
    """
    inp = Input(shape=(seq_daily, 1), name='input_ihsg')
    x = LSTM(lstm_units, return_sequences=True, name='lstm_1')(inp)
    x = Dropout(dropout_rate, name='drop_1')(x)
    x = LSTM(lstm_units // 2, return_sequences=False, name='lstm_2')(x)
    x = Dropout(dropout_rate, name='drop_2')(x)
    x = Dense(lstm_units // 4, activation='relu', name='dense_hidden')(x)
    x = Dropout(dropout_rate, name='drop_final')(x)
    output = Dense(1, activation='linear', name='output')(x)
    
    model = Model(inputs=inp, outputs=output, name='LSTM_Univariate')
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae']
    )
    return model


# Quick structure preview
m_sample = build_lstm_dual_branch(seq_daily=22, seq_monthly=3)
print("Dual-branch model summary:")
m_sample.summary()
tf.keras.backend.clear_session()
gc.collect()


Dual-branch model summary:


Model: "LSTM_DualBranch_Multivariate"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_daily         │ (None, 22, 2)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_monthly       │ (None, 3, 2)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_daily_1 (LSTM) │ (None, 22, 64)    │     17,152 │ input_daily[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_monthly_1      │ (None, 3, 64)     │     17,152 │ input_monthly[0]… │
│ (LSTM)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_daily_1        │ (None, 22, 64)    │          0 │ lstm_daily_1[0][… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_monthly_1      │ (None, 3, 64)     │          0 │ lstm_monthly_1[0… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_daily_2 (LSTM) │ (None, 32)        │     12,416 │ drop_daily_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_monthly_2      │ (None, 32)        │     12,416 │ drop_monthly_1[0… │
│ (LSTM)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_daily_2        │ (None, 32)        │          0 │ lstm_daily_2[0][… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_monthly_2      │ (None, 32)        │          0 │ lstm_monthly_2[0… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat              │ (None, 64)        │          0 │ drop_daily_2[0][… │
│ (Concatenate)       │                   │            │ drop_monthly_2[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_hidden        │ (None, 16)        │      1,040 │ concat[0][0]      │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_final          │ (None, 16)        │          0 │ dense_hidden[0][… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1)         │         17 │ drop_final[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 60,193 (235.13 KB)

 Trainable params: 60,193 (235.13 KB)

 Non-trainable params: 0 (0.00 B)

0

## 6. Helper: Metrics & Inverse Transform

In [15]:
def compute_metrics(y_true_scaled, y_pred_scaled, scaler_t):
    """
    Inverse transform dan hitung RMSE, MAPE, R² dalam skala asli (nilai IHSG).
    
    Parameters
    ----------
    y_true_scaled : np.array (N,)
    y_pred_scaled : np.array (N,)
    scaler_t      : fitted MinMaxScaler untuk target (IHSG saja)
    
    Returns
    -------
    dict dengan keys: rmse, mape, r2, y_true, y_pred  (skala asli)
    """
    y_true = scaler_t.inverse_transform(y_true_scaled.reshape(-1, 1)).flatten()
    y_pred = scaler_t.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
    r2   = r2_score(y_true, y_pred)
    
    return {"rmse": rmse, "mape": mape, "r2": r2,
            "y_true": y_true, "y_pred": y_pred}


def rmse_scaled(y_true_scaled, y_pred_scaled):
    """RMSE dalam skala [0,1] — dipakai saat grid search untuk konsistensi."""
    return np.sqrt(mean_squared_error(y_true_scaled, y_pred_scaled))


print("Helper functions defined: compute_metrics(), rmse_scaled()")


Helper functions defined: compute_metrics(), rmse_scaled()


## 7. Grid Search — Multivariate LSTM

Setiap kombinasi dilatih `GRID_HORIZON` epoch pada **train set** dengan val_split 15%.  
Metric utama: **val_loss (MSE)** → pilih yang terkecil.

Semua hasil disimpan ke `grid_results_multivariate.json` di `model/`.


In [16]:
grid_results_multi = []
best_multi         = {"val_loss": np.inf, "config": None}

# Semua kombinasi (kecuali seq_daily × seq_monthly yang tidak tersedia di sequences_dual)
available_keys = set(sequences_dual.keys())

# Buat semua kombinasi hyperparameter
hp_keys_seq  = [(sd, sm) for sd in GRID['seq_daily'] for sm in GRID['seq_monthly']
                 if (sd, sm) in available_keys]
hp_keys_rest = list(itertools.product(
    GRID['lstm_units'], GRID['dropout_rate'],
    GRID['batch_size'], GRID['learning_rate']
))
total_multi = len(hp_keys_seq) * len(hp_keys_rest)

print(f"Memulai grid search multivariate: {total_multi} kombinasi × {GRID_HORIZON} epoch")
print("=" * 75)

run_idx = 0
start_time = datetime.now()

for (seq_d, seq_m) in hp_keys_seq:
    seqs = sequences_dual[(seq_d, seq_m)]
    X_d_tr = seqs['X_daily_train']
    X_m_tr = seqs['X_monthly_train']
    y_tr   = seqs['y_train']
    
    for (units, drop, bs, lr) in hp_keys_rest:
        run_idx += 1
        
        tf.keras.backend.clear_session()
        gc.collect()
        
        model = build_lstm_dual_branch(
            seq_daily=seq_d, seq_monthly=seq_m,
            lstm_units=units, dropout_rate=drop, learning_rate=lr
        )
        
        history = model.fit(
            [X_d_tr, X_m_tr], y_tr,
            epochs=GRID_HORIZON,
            batch_size=bs,
            validation_split=VAL_SPLIT,
            verbose=0,
            shuffle=False
        )
        
        val_loss = min(history.history['val_loss'])
        
        config = {
            "seq_daily"     : seq_d,
            "seq_monthly"   : seq_m,
            "lstm_units"    : units,
            "dropout_rate"  : drop,
            "batch_size"    : bs,
            "learning_rate" : lr,
            "val_loss_proxy": round(float(val_loss), 8),
        }
        grid_results_multi.append(config)
        
        marker = " ◄ BEST" if val_loss < best_multi['val_loss'] else ""
        if val_loss < best_multi['val_loss']:
            best_multi = {"val_loss": val_loss, "config": config}
        
        elapsed = (datetime.now() - start_time).seconds
        print(f"[{run_idx:3d}/{total_multi}] d={seq_d:2d} m={seq_m:2d} "
              f"u={units:3d} dr={drop} bs={bs:2d} lr={lr} "
              f"→ val_loss={val_loss:.6f}{marker}  ({elapsed}s)")

print()
print("=" * 75)
print(f"✓ Grid search selesai")
print(f"  Best config: {best_multi['config']}")
print(f"  Best val_loss (proxy): {best_multi['val_loss']:.6f}")

# Simpan hasil grid
grid_results_multi_sorted = sorted(grid_results_multi, key=lambda x: x['val_loss_proxy'])
with open(MODEL_DIR / "grid_results_multivariate.json", "w") as f:
    json.dump({
        "best": best_multi['config'],
        "all_results": grid_results_multi_sorted
    }, f, indent=2)
print(f"\n✓ Hasil grid search disimpan → model/grid_results_multivariate.json")


Memulai grid search multivariate: 144 kombinasi × 5 epoch
[  1/144] d=22 m= 3 u= 64 dr=0.2 bs=16 lr=0.001 → val_loss=0.015428 ◄ BEST  (24s)
[  2/144] d=22 m= 3 u= 64 dr=0.2 bs=16 lr=0.0005 → val_loss=0.001443 ◄ BEST  (44s)
[  3/144] d=22 m= 3 u= 64 dr=0.2 bs=32 lr=0.001 → val_loss=0.007535  (63s)
[  4/144] d=22 m= 3 u= 64 dr=0.2 bs=32 lr=0.0005 → val_loss=0.016699  (80s)
[  5/144] d=22 m= 3 u= 64 dr=0.3 bs=16 lr=0.001 → val_loss=0.006950  (98s)
[  6/144] d=22 m= 3 u= 64 dr=0.3 bs=16 lr=0.0005 → val_loss=0.014846  (115s)
[  7/144] d=22 m= 3 u= 64 dr=0.3 bs=32 lr=0.001 → val_loss=0.010354  (131s)
[  8/144] d=22 m= 3 u= 64 dr=0.3 bs=32 lr=0.0005 → val_loss=0.023359  (148s)
[  9/144] d=22 m= 3 u=128 dr=0.2 bs=16 lr=0.001 → val_loss=0.003976  (171s)
[ 10/144] d=22 m= 3 u=128 dr=0.2 bs=16 lr=0.0005 → val_loss=0.001728  (195s)
[ 11/144] d=22 m= 3 u=128 dr=0.2 bs=32 lr=0.001 → val_loss=0.006508  (216s)
[ 12/144] d=22 m= 3 u=128 dr=0.2 bs=32 lr=0.0005 → val_loss=0.001365 ◄ BEST  (237s)
[ 13/144

## 8. Grid Search — Univariate LSTM (Baseline)

Prosedur grid search yang sama diterapkan pada model baseline univariate, tetapi hanya menyapu parameter yang relevan (`seq_daily`, `lstm_units`, `dropout_rate`, `batch_size`, `learning_rate`) karena model ini tidak memakai fitur bulanan. Hasil tiap kombinasi disimpan ke `grid_results_univariate.json`.


In [17]:
grid_results_uni = []
best_uni         = {"val_loss": np.inf, "config": None}

hp_uni_combos = list(itertools.product(
    GRID_UNI['seq_daily'], GRID_UNI['lstm_units'],
    GRID_UNI['dropout_rate'], GRID_UNI['batch_size'],
    GRID_UNI['learning_rate']
))
total_uni = len(hp_uni_combos)

print(f"Memulai grid search univariate: {total_uni} kombinasi × {GRID_HORIZON} epoch")
print("=" * 70)

run_idx   = 0
start_time = datetime.now()

for (seq_d, units, drop, bs, lr) in hp_uni_combos:
    run_idx += 1
    
    X_tr = sequences_uni[seq_d]['X_train']
    y_tr = sequences_uni[seq_d]['y_train']
    
    tf.keras.backend.clear_session()
    gc.collect()
    
    model = build_lstm_univariate(
        seq_daily=seq_d, lstm_units=units,
        dropout_rate=drop, learning_rate=lr
    )
    
    history = model.fit(
        X_tr, y_tr,
        epochs=GRID_HORIZON,
        batch_size=bs,
        validation_split=VAL_SPLIT,
        verbose=0,
        shuffle=False
    )
    
    val_loss = min(history.history['val_loss'])
    
    config = {
        "seq_daily"     : seq_d,
        "lstm_units"    : units,
        "dropout_rate"  : drop,
        "batch_size"    : bs,
        "learning_rate" : lr,
        "val_loss_proxy": round(float(val_loss), 8),
    }
    grid_results_uni.append(config)
    
    marker = " ◄ BEST" if val_loss < best_uni['val_loss'] else ""
    if val_loss < best_uni['val_loss']:
        best_uni = {"val_loss": val_loss, "config": config}
    
    elapsed = (datetime.now() - start_time).seconds
    print(f"[{run_idx:2d}/{total_uni}] d={seq_d:2d} u={units:3d} dr={drop} "
          f"bs={bs:2d} lr={lr} → val_loss={val_loss:.6f}{marker}  ({elapsed}s)")

print()
print("=" * 70)
print(f"✓ Grid search univariate selesai")
print(f"  Best config: {best_uni['config']}")

grid_results_uni_sorted = sorted(grid_results_uni, key=lambda x: x['val_loss_proxy'])
with open(MODEL_DIR / "grid_results_univariate.json", "w") as f:
    json.dump({
        "best": best_uni['config'],
        "all_results": grid_results_uni_sorted
    }, f, indent=2)
print(f"✓ Hasil grid search disimpan → model/grid_results_univariate.json")


Memulai grid search univariate: 48 kombinasi × 5 epoch
[ 1/48] d=22 u= 64 dr=0.2 bs=16 lr=0.001 → val_loss=0.001682 ◄ BEST  (44s)
[ 2/48] d=22 u= 64 dr=0.2 bs=16 lr=0.0005 → val_loss=0.001663 ◄ BEST  (78s)
[ 3/48] d=22 u= 64 dr=0.2 bs=32 lr=0.001 → val_loss=0.001362 ◄ BEST  (107s)
[ 4/48] d=22 u= 64 dr=0.2 bs=32 lr=0.0005 → val_loss=0.001530  (136s)
[ 5/48] d=22 u= 64 dr=0.3 bs=16 lr=0.001 → val_loss=0.005744  (169s)
[ 6/48] d=22 u= 64 dr=0.3 bs=16 lr=0.0005 → val_loss=0.002165  (207s)
[ 7/48] d=22 u= 64 dr=0.3 bs=32 lr=0.001 → val_loss=0.002241  (239s)
[ 8/48] d=22 u= 64 dr=0.3 bs=32 lr=0.0005 → val_loss=0.007456  (271s)
[ 9/48] d=22 u=128 dr=0.2 bs=16 lr=0.001 → val_loss=0.001806  (318s)
[10/48] d=22 u=128 dr=0.2 bs=16 lr=0.0005 → val_loss=0.001505  (364s)
[11/48] d=22 u=128 dr=0.2 bs=32 lr=0.001 → val_loss=0.001242 ◄ BEST  (401s)
[12/48] d=22 u=128 dr=0.2 bs=32 lr=0.0005 → val_loss=0.001402  (439s)
[13/48] d=22 u=128 dr=0.3 bs=16 lr=0.001 → val_loss=0.003082  (486s)
[14/48] d=22 u=1

## 9. Rangkuman Hasil Grid Search

In [18]:
print("=" * 65)
print("RANGKUMAN GRID SEARCH")
print("=" * 65)
print()
print("── MULTIVARIATE ─────────────────────────────────────────────")
for k, v in best_multi['config'].items():
    print(f"   {k:20s}: {v}")
print()
print("── UNIVARIATE ───────────────────────────────────────────────")
for k, v in best_uni['config'].items():
    print(f"   {k:20s}: {v}")
print()
print("Top-5 multivariate configs:")
for i, r in enumerate(grid_results_multi_sorted[:5], 1):
    print(f"  [{i}] d={r['seq_daily']:2d} m={r['seq_monthly']:2d} u={r['lstm_units']:3d} "
          f"dr={r['dropout_rate']} bs={r['batch_size']:2d} lr={r['learning_rate']} "
          f"→ val_loss={r['val_loss_proxy']:.6f}")
print()
print("Top-5 univariate configs:")
for i, r in enumerate(grid_results_uni_sorted[:5], 1):
    print(f"  [{i}] d={r['seq_daily']:2d} u={r['lstm_units']:3d} "
          f"dr={r['dropout_rate']} bs={r['batch_size']:2d} lr={r['learning_rate']} "
          f"→ val_loss={r['val_loss_proxy']:.6f}")


RANGKUMAN GRID SEARCH

── MULTIVARIATE ─────────────────────────────────────────────
   seq_daily           : 90
   seq_monthly         : 3
   lstm_units          : 128
   dropout_rate        : 0.2
   batch_size          : 32
   learning_rate       : 0.0005
   val_loss_proxy      : 0.00130695

── UNIVARIATE ───────────────────────────────────────────────
   seq_daily           : 90
   lstm_units          : 128
   dropout_rate        : 0.2
   batch_size          : 32
   learning_rate       : 0.001
   val_loss_proxy      : 0.00119422

Top-5 multivariate configs:
  [1] d=90 m= 3 u=128 dr=0.2 bs=32 lr=0.0005 → val_loss=0.001307
  [2] d=90 m= 6 u=128 dr=0.2 bs=32 lr=0.001 → val_loss=0.001330
  [3] d=60 m= 3 u=128 dr=0.2 bs=32 lr=0.0005 → val_loss=0.001353
  [4] d=90 m=12 u=128 dr=0.3 bs=32 lr=0.001 → val_loss=0.001361
  [5] d=22 m= 3 u=128 dr=0.2 bs=32 lr=0.0005 → val_loss=0.001365

Top-5 univariate configs:
  [1] d=90 u=128 dr=0.2 bs=32 lr=0.001 → val_loss=0.001194
  [2] d=60 u=128 dr=0.2 

## 10. Final Training — Multivariate LSTM

Train ulang dengan best config dari grid search, menggunakan:
- Epoch penuh (max `FINAL_EPOCHS`)
- EarlyStopping (restore best weights)
- ReduceLROnPlateau
- ModelCheckpoint → simpan best model saat training


In [31]:
# ── Load best config ──
bc_m = best_multi['config']
# Override seq_monthly berdasarkan evaluasi penuh (grid search underestimate karena 5 epoch)
bc_m['seq_monthly'] = 12

print("Best multivariate config:")
for k, v in bc_m.items():
    if k != 'val_loss_proxy':
        print(f"  {k}: {v}")

# ── Ambil sequences dengan best (seq_daily, seq_monthly) ──
key_m = (bc_m['seq_daily'], bc_m['seq_monthly'])
seqs_m = sequences_dual[key_m]

X_d_tr_final = seqs_m['X_daily_train']
X_m_tr_final = seqs_m['X_monthly_train']
y_tr_final   = seqs_m['y_train']
X_d_te_final = seqs_m['X_daily_test']
X_m_te_final = seqs_m['X_monthly_test']
y_te_final   = seqs_m['y_test']

print(f"\nTrain: X_daily={X_d_tr_final.shape}, X_monthly={X_m_tr_final.shape}, y={y_tr_final.shape}")
print(f"Test : X_daily={X_d_te_final.shape}, X_monthly={X_m_te_final.shape}, y={y_te_final.shape}")


Best multivariate config:
  seq_daily: 90
  seq_monthly: 12
  lstm_units: 128
  dropout_rate: 0.2
  batch_size: 32
  learning_rate: 0.0005

Train: X_daily=(1682, 90, 2), X_monthly=(1682, 12, 2), y=(1682,)
Test : X_daily=(485, 90, 2), X_monthly=(485, 12, 2), y=(485,)


In [32]:
tf.keras.backend.clear_session()
gc.collect()

model_multi = build_lstm_dual_branch(
    seq_daily=bc_m['seq_daily'],
    seq_monthly=bc_m['seq_monthly'],
    lstm_units=bc_m['lstm_units'],
    dropout_rate=bc_m['dropout_rate'],
    learning_rate=bc_m['learning_rate'],
)

callbacks_multi = [
    EarlyStopping(
        monitor='val_loss', patience=PATIENCE_ES,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=PATIENCE_LR, min_lr=1e-6, verbose=1
    ),
    ModelCheckpoint(
        filepath=str(MODEL_DIR / "best_multi_checkpoint.keras"),
        monitor='val_loss', save_best_only=True, verbose=0
    ),
]

print(f"Final training multivariate — max {FINAL_EPOCHS} epoch...")
history_multi = model_multi.fit(
    [X_d_tr_final, X_m_tr_final], y_tr_final,
    epochs=FINAL_EPOCHS,
    batch_size=bc_m['batch_size'],
    validation_split=VAL_SPLIT,
    callbacks=callbacks_multi,
    shuffle=False,
    verbose=1
)

epochs_ran = len(history_multi.history['loss'])
print(f"\n✓ Training selesai — {epochs_ran} epoch aktual")
print(f"  Best val_loss : {min(history_multi.history['val_loss']):.6f}")


Final training multivariate — max 100 epoch...
Epoch 1/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 57s 465ms/step - loss: 0.0359 - mae: 0.1575 - val_loss: 0.0215 - val_mae: 0.1406 - learning_rate: 5.0000e-04
Epoch 2/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 34s 328ms/step - loss: 0.0388 - mae: 0.1465 - val_loss: 0.0018 - val_mae: 0.0334 - learning_rate: 5.0000e-04
Epoch 3/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 20s 309ms/step - loss: 0.0131 - mae: 0.0879 - val_loss: 0.0050 - val_mae: 0.0634 - learning_rate: 5.0000e-04
Epoch 4/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 14s 301ms/step - loss: 0.0185 - mae: 0.1098 - val_loss: 0.0110 - val_mae: 0.0988 - learning_rate: 5.0000e-04
Epoch 5/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 13s 296ms/step - loss: 0.0121 - mae: 0.0848 - val_loss: 0.0033 - val_mae: 0.0496 - learning_rate: 5.0000e-04
Epoch 6/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 14s 305ms/step - loss: 0.0108 - mae: 0.0793 - val_loss: 0.0059 - val_mae: 0.0707 - learning_rate: 5.0000e-04
Epoch 7/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 14s 308ms/step - loss: 0.0

## 11. Final Training — Univariate LSTM (Baseline)

Model baseline univariate dilatih ulang dengan konfigurasi terbaik hasil grid search, memakai epoch penuh plus EarlyStopping (restore best weights), ReduceLROnPlateau, dan ModelCheckpoint — prosedur yang sama dengan final training multivariate agar perbandingan kedua model adil.


In [21]:
bc_u = best_uni['config']
print("Best univariate config:")
for k, v in bc_u.items():
    if k != 'val_loss_proxy':
        print(f"  {k}: {v}")

X_tr_uni_final = sequences_uni[bc_u['seq_daily']]['X_train']
y_tr_uni_final = sequences_uni[bc_u['seq_daily']]['y_train']
X_te_uni_final = sequences_uni[bc_u['seq_daily']]['X_test']
y_te_uni_final = sequences_uni[bc_u['seq_daily']]['y_test']

print(f"\nTrain: X={X_tr_uni_final.shape}, y={y_tr_uni_final.shape}")
print(f"Test : X={X_te_uni_final.shape}, y={y_te_uni_final.shape}")


Best univariate config:
  seq_daily: 90
  lstm_units: 128
  dropout_rate: 0.2
  batch_size: 32
  learning_rate: 0.001

Train: X=(1849, 90, 1), y=(1849,)
Test : X=(485, 90, 1), y=(485,)


In [22]:
tf.keras.backend.clear_session()
gc.collect()

model_uni = build_lstm_univariate(
    seq_daily=bc_u['seq_daily'],
    lstm_units=bc_u['lstm_units'],
    dropout_rate=bc_u['dropout_rate'],
    learning_rate=bc_u['learning_rate'],
)

callbacks_uni = [
    EarlyStopping(
        monitor='val_loss', patience=PATIENCE_ES,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=PATIENCE_LR, min_lr=1e-6, verbose=1
    ),
    ModelCheckpoint(
        filepath=str(MODEL_DIR / "best_uni_checkpoint.keras"),
        monitor='val_loss', save_best_only=True, verbose=0
    ),
]

print(f"Final training univariate — max {FINAL_EPOCHS} epoch...")
history_uni = model_uni.fit(
    X_tr_uni_final, y_tr_uni_final,
    epochs=FINAL_EPOCHS,
    batch_size=bc_u['batch_size'],
    validation_split=VAL_SPLIT,
    callbacks=callbacks_uni,
    shuffle=False,
    verbose=1
)

epochs_ran_u = len(history_uni.history['loss'])
print(f"\n✓ Training selesai — {epochs_ran_u} epoch aktual")
print(f"  Best val_loss : {min(history_uni.history['val_loss']):.6f}")


Final training univariate — max 100 epoch...
Epoch 1/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 309ms/step - loss: 0.0223 - mae: 0.1148 - val_loss: 0.0016 - val_mae: 0.0320 - learning_rate: 0.0010
Epoch 2/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 18s 265ms/step - loss: 0.0286 - mae: 0.1400 - val_loss: 0.0301 - val_mae: 0.1682 - learning_rate: 0.0010
Epoch 3/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 20s 263ms/step - loss: 0.0179 - mae: 0.1073 - val_loss: 0.0030 - val_mae: 0.0466 - learning_rate: 0.0010
Epoch 4/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 13s 252ms/step - loss: 0.0122 - mae: 0.0864 - val_loss: 0.0013 - val_mae: 0.0264 - learning_rate: 0.0010
Epoch 5/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 261ms/step - loss: 0.0097 - mae: 0.0768 - val_loss: 0.0040 - val_mae: 0.0569 - learning_rate: 0.0010
Epoch 6/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 265ms/step - loss: 0.0081 - mae: 0.0693 - val_loss: 0.0031 - val_mae: 0.0491 - learning_rate: 0.0010
Epoch 7/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 20s 262ms/step - loss: 0.0098 - mae: 0.0758 - val_lo

## 12. Evaluasi pada Test Set

Prediksi pada test set, inverse transform ke skala IHSG asli, lalu hitung:
- **RMSE** — Root Mean Squared Error (satuan: poin IHSG)
- **MAPE** — Mean Absolute Percentage Error (%)
- **R²**   — Coefficient of Determination


In [40]:
# Load bobot terbaik dari checkpoint
model_multi = tf.keras.models.load_model(str(MODEL_DIR / "best_multi_checkpoint.keras"))
model_uni   = tf.keras.models.load_model(str(MODEL_DIR / "best_uni_checkpoint.keras"))
print("✓ Best checkpoint loaded untuk kedua model")

# Fix: y_test konsisten untuk semua seq_daily=90, ambil dari (90,3) yang range-nya benar
y_te_final = sequences_dual[(90, 3)]['y_test']

# ── Prediksi Multivariate ──
y_pred_multi_scaled = model_multi.predict(
    [X_d_te_final, X_m_te_final], verbose=0
).flatten()

metrics_multi = compute_metrics(y_te_final, y_pred_multi_scaled, scaler_target)

# ── Prediksi Univariate ──
y_pred_uni_scaled = model_uni.predict(X_te_uni_final, verbose=0).flatten()
metrics_uni = compute_metrics(y_te_uni_final, y_pred_uni_scaled, scaler_uni)

# ── Tabel Perbandingan ──
print("=" * 55)
print(f"{'Metrik':<12} {'Univariate':>15} {'Multivariate':>15}")
print("-" * 55)
print(f"{'RMSE':<12} {metrics_uni['rmse']:>15.2f} {metrics_multi['rmse']:>15.2f}")
print(f"{'MAPE (%)':<12} {metrics_uni['mape']:>15.4f} {metrics_multi['mape']:>15.4f}")
print(f"{'R²':<12} {metrics_uni['r2']:>15.4f} {metrics_multi['r2']:>15.4f}")
print("=" * 55)

delta_rmse = metrics_uni['rmse'] - metrics_multi['rmse']
delta_mape = metrics_uni['mape'] - metrics_multi['mape']
delta_r2   = metrics_multi['r2'] - metrics_uni['r2']

print(f"\nDelta (Uni − Multi):")
print(f"  RMSE improvement : {delta_rmse:+.2f} poin IHSG ({'Multivariate lebih baik' if delta_rmse > 0 else 'Univariate lebih baik'})")
print(f"  MAPE improvement : {delta_mape:+.4f}%  ({'Multivariate lebih baik' if delta_mape > 0 else 'Univariate lebih baik'})")
print(f"  R²   improvement : {delta_r2:+.4f}  ({'Multivariate lebih baik' if delta_r2 > 0 else 'Univariate lebih baik'})")


✓ Best checkpoint loaded untuk kedua model
Metrik            Univariate    Multivariate
-------------------------------------------------------
RMSE                  122.41          440.10
MAPE (%)              1.3289          5.2400
R²                    0.8283         -1.2198

Delta (Uni − Multi):
  RMSE improvement : -317.68 poin IHSG (Univariate lebih baik)
  MAPE improvement : -3.9111%  (Univariate lebih baik)
  R²   improvement : -2.0481  (Univariate lebih baik)


## 12a. Diagnostik & Sanity Check Hasil Evaluasi

Sebelum model disimpan, sel-sel berikut melakukan pemeriksaan diagnostik untuk memastikan hasil evaluasi valid dan tidak ada anomali pada data maupun prediksi. Pengecekan mencakup statistik (min/max/mean) nilai target dan prediksi, konsistensi `history` pelatihan, kesesuaian bobot *checkpoint* dengan model di memori, rentang `scaler`, serta variansi fitur bulanan (inflasi dan BI Rate).


In [34]:
print("y_te_final stats:")
print(f"  min: {y_te_final.min():.4f}, max: {y_te_final.max():.4f}, mean: {y_te_final.mean():.4f}")

print("\ny_pred_multi_scaled stats:")
print(f"  min: {y_pred_multi_scaled.min():.4f}, max: {y_pred_multi_scaled.max():.4f}, mean: {y_pred_multi_scaled.mean():.4f}")

print("\ny_te_uni_final stats:")
print(f"  min: {y_te_uni_final.min():.4f}, max: {y_te_uni_final.max():.4f}, mean: {y_te_uni_final.mean():.4f}")

print("\ny_pred_uni_scaled stats:")
print(f"  min: {y_pred_uni_scaled.min():.4f}, max: {y_pred_uni_scaled.max():.4f}, mean: {y_pred_uni_scaled.mean():.4f}")

y_te_final stats:
  min: 0.7775, max: 1.1738, mean: 0.9297

y_pred_multi_scaled stats:
  min: 0.7016, max: 0.9669, mean: 0.8218

y_te_uni_final stats:
  min: 0.7775, max: 1.1738, mean: 0.9297

y_pred_uni_scaled stats:
  min: 0.7986, max: 1.0908, mean: 0.9103


In [35]:
print("Training history multivariate:")
print(f"  Epochs ran     : {len(history_multi.history['val_loss'])}")
print(f"  Best val_loss  : {min(history_multi.history['val_loss']):.6f}")
print(f"  Final val_loss : {history_multi.history['val_loss'][-1]:.6f}")
print(f"  val_loss epoch 1-10: {history_multi.history['val_loss'][:10]}")

Training history multivariate:
  Epochs ran     : 29
  Best val_loss  : 0.001230
  Final val_loss : 0.001277
  val_loss epoch 1-10: [0.02151654101908207, 0.0018120442982763052, 0.004990880843251944, 0.010957219637930393, 0.003279592376202345, 0.00592493312433362, 0.0029963860288262367, 0.001680515008047223, 0.0019710783381015062, 0.0033934807870537043]


In [36]:
# Cek apakah checkpoint yang di-load setelah training adalah yang terbaik
model_loaded = tf.keras.models.load_model(str(MODEL_DIR / "best_multi_checkpoint.keras"))
y_pred_check = model_loaded.predict([X_d_te_final, X_m_te_final], verbose=0).flatten()
metrics_check = compute_metrics(y_te_final, y_pred_check, scaler_target)
print(f"Dari checkpoint file → RMSE: {metrics_check['rmse']:.2f}, R²: {metrics_check['r2']:.4f}")

# Bandingkan dengan model di memori
y_pred_mem = model_multi.predict([X_d_te_final, X_m_te_final], verbose=0).flatten()
metrics_mem = compute_metrics(y_te_final, y_pred_mem, scaler_target)
print(f"Dari model di memori  → RMSE: {metrics_mem['rmse']:.2f}, R²: {metrics_mem['r2']:.4f}")

Dari checkpoint file → RMSE: 440.10, R²: -1.2198
Dari model di memori  → RMSE: 440.10, R²: -1.2198


In [37]:
print("scaler_target range:")
print(f"  data_min_ : {scaler_target.data_min_}")
print(f"  data_max_ : {scaler_target.data_max_}")
print(f"  scale_    : {scaler_target.scale_}")

print("\nscaler_uni range:")
print(f"  data_min_ : {scaler_uni.data_min_}")
print(f"  data_max_ : {scaler_uni.data_max_}")
print(f"  scale_    : {scaler_uni.scale_}")

# Cek juga nilai IHSG asli dari y_te_final setelah inverse
y_te_inv_target = scaler_target.inverse_transform(y_te_final.reshape(-1,1)).flatten()
y_te_inv_uni    = scaler_uni.inverse_transform(y_te_uni_final.reshape(-1,1)).flatten()
print(f"\ny_te_final inverse via scaler_target: min={y_te_inv_target.min():.0f}, max={y_te_inv_target.max():.0f}")
print(f"y_te_uni_final inverse via scaler_uni: min={y_te_inv_uni.min():.0f}, max={y_te_inv_uni.max():.0f}")

scaler_target range:
  data_min_ : [3937.63]
  data_max_ : [7318.02]
  scale_    : [0.00029582]

scaler_uni range:
  data_min_ : [3937.63]
  data_max_ : [7318.02]
  scale_    : [0.00029582]

y_te_final inverse via scaler_target: min=6566, max=7905
y_te_uni_final inverse via scaler_uni: min=6566, max=7905


In [38]:
# Dari sequences mana y_te_final berasal sekarang?
print(f"y_te_final shape  : {y_te_final.shape}")
print(f"y_tr_final shape  : {y_tr_final.shape}")
print(f"key_m yang dipakai: {key_m}")

# Bandingkan dengan y_test dari key (90,3) yang lama
y_te_lama = sequences_dual[(90, 3)]['y_test']
print(f"\ny_test (90,3) : min={y_te_lama.min():.4f}, max={y_te_lama.max():.0f}")
print(f"y_te_final    : min={y_te_final.min():.4f}, max={y_te_final.max():.4f}")

y_te_final shape  : (485,)
y_tr_final shape  : (1682,)
key_m yang dipakai: (90, 12)

y_test (90,3) : min=0.7775, max=1
y_te_final    : min=0.7775, max=1.1738


In [41]:
print(f"y_pred_multi_scaled: min={y_pred_multi_scaled.min():.4f}, max={y_pred_multi_scaled.max():.4f}, mean={y_pred_multi_scaled.mean():.4f}")
print(f"y_te_final         : min={y_te_final.min():.4f}, max={y_te_final.max():.4f}, mean={y_te_final.mean():.4f}")
print(f"\ny_pred_uni_scaled  : min={y_pred_uni_scaled.min():.4f}, max={y_pred_multi_scaled.max():.4f}, mean={y_pred_uni_scaled.mean():.4f}")
print(f"y_te_uni_final     : min={y_te_uni_final.min():.4f}, max={y_te_uni_final.max():.4f}, mean={y_te_uni_final.mean():.4f}")

y_pred_multi_scaled: min=0.7016, max=0.9669, mean=0.8218
y_te_final         : min=0.7775, max=1.1738, mean=0.9297

y_pred_uni_scaled  : min=0.7986, max=0.9669, mean=0.9103
y_te_uni_final     : min=0.7775, max=1.1738, mean=0.9297


In [42]:
# Force predict ulang keduanya dari model yang ada di memori sekarang
y_pred_multi_check = model_multi.predict([X_d_te_final, X_m_te_final], verbose=0).flatten()
y_pred_uni_check   = model_uni.predict(X_te_uni_final, verbose=0).flatten()

print(f"multi: min={y_pred_multi_check.min():.4f}, max={y_pred_multi_check.max():.4f}, mean={y_pred_multi_check.mean():.4f}")
print(f"uni  : min={y_pred_uni_check.min():.4f},  max={y_pred_uni_check.max():.4f},  mean={y_pred_uni_check.mean():.4f}")

# Bandingkan dengan y_true
print(f"true : min={y_te_final.min():.4f},  max={y_te_final.max():.4f},  mean={y_te_final.mean():.4f}")

multi: min=0.7016, max=0.9669, mean=0.8218
uni  : min=0.7986,  max=1.0908,  mean=0.9103
true : min=0.7775,  max=1.1738,  mean=0.9297


In [45]:
# Cek apakah monthly features punya variansi yang cukup
seqs_12 = sequences_dual[(90, 12)]
X_m_tr = seqs_12['X_monthly_train']
X_m_te = seqs_12['X_monthly_test']

print(f"X_monthly_train shape : {X_m_tr.shape}")
print(f"X_monthly_test  shape : {X_m_te.shape}")
print(f"\nX_monthly_train — min: {X_m_tr.min():.4f}, max: {X_m_tr.max():.4f}, mean: {X_m_tr.mean():.4f}, std: {X_m_tr.std():.4f}")
print(f"X_monthly_test  — min: {X_m_te.min():.4f}, max: {X_m_te.max():.4f}, mean: {X_m_te.mean():.4f}, std: {X_m_te.std():.4f}")

# Cek per fitur (inflasi vs bi_rate)
print(f"\nInflasi  train — min: {X_m_tr[:,:,0].min():.4f}, max: {X_m_tr[:,:,0].max():.4f}, std: {X_m_tr[:,:,0].std():.4f}")
print(f"BI Rate  train — min: {X_m_tr[:,:,1].min():.4f}, max: {X_m_tr[:,:,1].max():.4f}, std: {X_m_tr[:,:,1].std():.4f}")
print(f"\nInflasi  test  — min: {X_m_te[:,:,0].min():.4f}, max: {X_m_te[:,:,0].max():.4f}, std: {X_m_te[:,:,0].std():.4f}")
print(f"BI Rate  test  — min: {X_m_te[:,:,1].min():.4f}, max: {X_m_te[:,:,1].max():.4f}, std: {X_m_te[:,:,1].std():.4f}")

X_monthly_train shape : (1682, 12, 2)
X_monthly_test  shape : (485, 12, 2)

X_monthly_train — min: 0.0000, max: 1.0000, mean: 0.3897, std: 0.2429
X_monthly_test  — min: 0.0000, max: 1.0000, mean: 0.4483, std: 0.1904

Inflasi  train — min: 0.0000, max: 1.0000, std: 0.1781
BI Rate  train — min: 0.0000, max: 1.0000, std: 0.2881

Inflasi  test  — min: 0.1481, max: 1.0000, std: 0.1799
BI Rate  test  — min: 0.0000, max: 0.6471, std: 0.1995


## 12b. Pelatihan Ulang Multivariate (Verifikasi Kestabilan)

Model multivariate dilatih ulang dengan konfigurasi yang disesuaikan — *learning rate* lebih kecil (`1e-4`), `patience` dinaikkan menjadi 20, dan `seq_monthly=12` — untuk memverifikasi apakah hasil evaluasi stabil dan pelatihan tidak berhenti terlalu dini. Metriknya kemudian dibandingkan terhadap model utama.


In [43]:
tf.keras.backend.clear_session()
gc.collect()

model_multi = build_lstm_dual_branch(
    seq_daily=90,
    seq_monthly=12,
    lstm_units=128,
    dropout_rate=0.2,
    learning_rate=0.0001,  # lebih kecil dari 0.0005
)

callbacks_multi = [
    EarlyStopping(
        monitor='val_loss', patience=20,  # lebih sabar
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=10, min_lr=1e-7, verbose=1
    ),
    ModelCheckpoint(
        filepath=str(MODEL_DIR / "best_multi_checkpoint.keras"),
        monitor='val_loss', save_best_only=True, verbose=0
    ),
]

print("Retrain multivariate — lr=0.0001, patience=20...")
history_multi = model_multi.fit(
    [X_d_tr_final, X_m_tr_final], y_tr_final,
    epochs=150,
    batch_size=32,
    validation_split=0.15,
    callbacks=callbacks_multi,
    shuffle=False,
    verbose=1
)

epochs_ran = len(history_multi.history['loss'])
print(f"\n✓ Training selesai — {epochs_ran} epoch aktual")
print(f"  Best val_loss : {min(history_multi.history['val_loss']):.6f}")

Retrain multivariate — lr=0.0001, patience=20...
Epoch 1/150
45/45 ━━━━━━━━━━━━━━━━━━━━ 40s 388ms/step - loss: 0.1002 - mae: 0.2781 - val_loss: 0.0420 - val_mae: 0.1992 - learning_rate: 1.0000e-04
Epoch 2/150
45/45 ━━━━━━━━━━━━━━━━━━━━ 17s 310ms/step - loss: 0.0411 - mae: 0.1622 - val_loss: 0.0448 - val_mae: 0.2066 - learning_rate: 1.0000e-04
Epoch 3/150
45/45 ━━━━━━━━━━━━━━━━━━━━ 21s 312ms/step - loss: 0.0273 - mae: 0.1303 - val_loss: 0.0307 - val_mae: 0.1693 - learning_rate: 1.0000e-04
Epoch 4/150
45/45 ━━━━━━━━━━━━━━━━━━━━ 21s 321ms/step - loss: 0.0246 - mae: 0.1235 - val_loss: 0.0127 - val_mae: 0.1044 - learning_rate: 1.0000e-04
Epoch 5/150
45/45 ━━━━━━━━━━━━━━━━━━━━ 14s 314ms/step - loss: 0.0197 - mae: 0.1104 - val_loss: 0.0082 - val_mae: 0.0818 - learning_rate: 1.0000e-04
Epoch 6/150
45/45 ━━━━━━━━━━━━━━━━━━━━ 21s 315ms/step - loss: 0.0184 - mae: 0.1050 - val_loss: 0.0025 - val_mae: 0.0416 - learning_rate: 1.0000e-04
Epoch 7/150
45/45 ━━━━━━━━━━━━━━━━━━━━ 15s 324ms/step - loss: 0

In [46]:
y_pred_multi_check = model_multi.predict([X_d_te_final, X_m_te_final], verbose=0).flatten()
metrics_check = compute_metrics(y_te_final, y_pred_multi_check, scaler_target)
print(f"RMSE: {metrics_check['rmse']:.2f}, MAPE: {metrics_check['mape']:.4f}%, R²: {metrics_check['r2']:.4f}")
print(f"pred: min={y_pred_multi_check.min():.4f}, max={y_pred_multi_check.max():.4f}, mean={y_pred_multi_check.mean():.4f}")
print(f"true: min={y_te_final.min():.4f},         max={y_te_final.max():.4f},         mean={y_te_final.mean():.4f}")

RMSE: 167.06, MAPE: 1.8919%, R²: 0.6801
pred: min=0.8279, max=1.0645, mean=0.9234
true: min=0.7775,         max=1.1738,         mean=0.9297


## 12c. Perbandingan Panjang Sekuens Bulanan (seq_monthly 3 vs 12)

Eksperimen pembanding untuk memastikan panjang sekuens bulanan yang dipilih sudah tepat: konfigurasi terbaik dilatih ulang dengan `seq_monthly=12`, lalu metriknya (RMSE, MAPE, R²) dibandingkan langsung terhadap `seq_monthly=3`.


In [28]:
# Cek: retrain dengan seq_monthly=12 dan bandingkan
from copy import deepcopy

bc_m_test = deepcopy(bc_m)
bc_m_test['seq_monthly'] = 12

key_test = (bc_m_test['seq_daily'], 12)
seqs_test = sequences_dual[key_test]

X_d_tr_t = seqs_test['X_daily_train']
X_m_tr_t = seqs_test['X_monthly_train']
y_tr_t   = seqs_test['y_train']
X_d_te_t = seqs_test['X_daily_test']
X_m_te_t = seqs_test['X_monthly_test']
y_te_t   = seqs_test['y_test']

model_test = build_lstm_dual_branch(
    seq_daily=bc_m_test['seq_daily'],
    seq_monthly=12,
    lstm_units=bc_m_test['lstm_units'],
    dropout_rate=bc_m_test['dropout_rate'],
    learning_rate=bc_m_test['learning_rate']
)

cb_test = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6)
]

history_test = model_test.fit(
    [X_d_tr_t, X_m_tr_t], y_tr_t,
    validation_split=0.15,
    epochs=100,
    batch_size=bc_m_test['batch_size'],
    callbacks=cb_test,
    verbose=1
)

y_pred_test_scaled = model_test.predict([X_d_te_t, X_m_te_t], verbose=0).flatten()
metrics_test = compute_metrics(y_te_t, y_pred_test_scaled, scaler_target)
print(f"\nseq_monthly=12 → RMSE: {metrics_test['rmse']:.2f}, MAPE: {metrics_test['mape']:.4f}%, R²: {metrics_test['r2']:.4f}")
print(f"seq_monthly= 3 → RMSE: {metrics_multi['rmse']:.2f}, MAPE: {metrics_multi['mape']:.4f}%, R²: {metrics_multi['r2']:.4f}")

Epoch 1/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 34s 358ms/step - loss: 0.0512 - mae: 0.1704 - val_loss: 0.0290 - val_mae: 0.1651 - learning_rate: 5.0000e-04
Epoch 2/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 14s 302ms/step - loss: 0.0138 - mae: 0.0908 - val_loss: 0.0090 - val_mae: 0.0879 - learning_rate: 5.0000e-04
Epoch 3/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 21s 302ms/step - loss: 0.0115 - mae: 0.0833 - val_loss: 0.0033 - val_mae: 0.0508 - learning_rate: 5.0000e-04
Epoch 4/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 13s 300ms/step - loss: 0.0099 - mae: 0.0765 - val_loss: 0.0024 - val_mae: 0.0411 - learning_rate: 5.0000e-04
Epoch 5/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 21s 306ms/step - loss: 0.0087 - mae: 0.0711 - val_loss: 0.0019 - val_mae: 0.0343 - learning_rate: 5.0000e-04
Epoch 6/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 14s 309ms/step - loss: 0.0095 - mae: 0.0749 - val_loss: 0.0031 - val_mae: 0.0472 - learning_rate: 5.0000e-04
Epoch 7/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 13s 292ms/step - loss: 0.0092 - mae: 0.0737 - val_loss: 0.0027 - val_mae:

In [27]:
print(sequences_dual[(90, 12)].keys())

dict_keys(['X_daily_train', 'X_monthly_train', 'y_train', 'dates_train', 'X_daily_test', 'X_monthly_test', 'y_test', 'dates_test'])


## 13. Simpan Model & Artefak Lengkap

### Artefak yang disimpan

| File | Keterangan |
|---|---|
| `model_multivariate.keras` | Model multivariate final (Keras native format) |
| `model_univariate.keras` | Model univariate/baseline final |
| `best_multi_checkpoint.keras` | Checkpoint best val_loss multivariate |
| `best_uni_checkpoint.keras` | Checkpoint best val_loss univariate |
| `model_config.json` | Config lengkap kedua model + metrik evaluasi |
| `training_history_multi.json` | History loss multivariate per epoch |
| `training_history_uni.json` | History loss univariate per epoch |
| `predictions_test.joblib` | y_true, y_pred, tanggal test set (kedua model) |
| `grid_results_multivariate.json` | Semua hasil grid search multivariate |
| `grid_results_univariate.json` | Semua hasil grid search univariate |

**Format `.keras`** dipilih karena fully serializable, tidak bergantung TF version,
dan bisa di-load kembali hanya dengan `tf.keras.models.load_model('path.keras')`.


In [47]:
# ── 1. Simpan model final ──
model_multi.save(MODEL_DIR / "model_multivariate.keras")
model_uni.save(MODEL_DIR / "model_univariate.keras")
print("✓ model_multivariate.keras")
print("✓ model_univariate.keras")

# ── 2. Config lengkap ──
model_config = {
    "created_at"    : datetime.now().isoformat(),
    "multivariate"  : {
        "architecture"  : "Dual-Branch LSTM (Keras Functional API)",
        "hyperparameters": {k: v for k, v in bc_m.items() if k != 'val_loss_proxy'},
        "grid_val_loss_proxy": bc_m['val_loss_proxy'],
        "final_epochs"  : len(history_multi.history['loss']),
        "final_val_loss": float(min(history_multi.history['val_loss'])),
        "test_metrics"  : {
            "rmse": round(float(metrics_multi['rmse']), 4),
            "mape": round(float(metrics_multi['mape']), 4),
            "r2"  : round(float(metrics_multi['r2']),   4),
        },
        "input_daily_shape"   : list(X_d_te_final.shape[1:]),
        "input_monthly_shape" : list(X_m_te_final.shape[1:]),
        "daily_features"      : DAILY_FEATURES,
        "monthly_features"    : MONTHLY_FEATURES,
        "target_col"          : TARGET_COL,
        "scaler_files"        : {
            "daily"  : "data/processed/scaler_daily.joblib",
            "monthly": "data/processed/scaler_monthly.joblib",
            "target" : "data/processed/scaler_target.joblib",
        }
    },
    "univariate"  : {
        "architecture"  : "Single-Branch LSTM",
        "hyperparameters": {k: v for k, v in bc_u.items() if k != 'val_loss_proxy'},
        "grid_val_loss_proxy": bc_u['val_loss_proxy'],
        "final_epochs"  : len(history_uni.history['loss']),
        "final_val_loss": float(min(history_uni.history['val_loss'])),
        "test_metrics"  : {
            "rmse": round(float(metrics_uni['rmse']), 4),
            "mape": round(float(metrics_uni['mape']), 4),
            "r2"  : round(float(metrics_uni['r2']),   4),
        },
        "input_shape" : list(X_te_uni_final.shape[1:]),
        "features"    : ["ihsg"],
        "target_col"  : TARGET_COL,
        "scaler_file" : "data/processed/scaler_univariate.joblib"
    },
    "comparison"  : {
        "rmse_delta"       : round(float(delta_rmse), 4),
        "mape_delta"       : round(float(delta_mape), 4),
        "r2_delta"         : round(float(delta_r2), 4),
        "winner_rmse"      : "multivariate" if delta_rmse > 0 else "univariate",
        "winner_mape"      : "multivariate" if delta_mape > 0 else "univariate",
        "winner_r2"        : "multivariate" if delta_r2 > 0 else "univariate",
    }
}

with open(MODEL_DIR / "model_config.json", "w") as f:
    json.dump(model_config, f, indent=2)
print("✓ model_config.json")

# ── 3. Training histories ──
with open(MODEL_DIR / "training_history_multi.json", "w") as f:
    json.dump({k: [float(x) for x in v]
               for k, v in history_multi.history.items()}, f, indent=2)
with open(MODEL_DIR / "training_history_uni.json", "w") as f:
    json.dump({k: [float(x) for x in v]
               for k, v in history_uni.history.items()}, f, indent=2)
print("✓ training_history_multi.json")
print("✓ training_history_uni.json")

# ── 4. Predictions ──
# Simpan tanggal test set
dates_test_multi = seqs_m['dates_test']
dates_test_uni   = [None] * len(y_te_uni_final)  # univariate tidak track tanggal per sequence

predictions_data = {
    "multivariate": {
        "dates"  : [str(pd.Timestamp(d).date()) if d is not None else None for d in dates_test_multi],
        "y_true" : metrics_multi['y_true'].tolist(),
        "y_pred" : metrics_multi['y_pred'].tolist(),
    },
    "univariate": {
        "y_true" : metrics_uni['y_true'].tolist(),
        "y_pred" : metrics_uni['y_pred'].tolist(),
    }
}
joblib.dump(predictions_data, MODEL_DIR / "predictions_test.joblib")
print("✓ predictions_test.joblib")

print()
print("=" * 60)
print("SEMUA ARTEFAK MODEL TERSIMPAN:")
for p in sorted(MODEL_DIR.iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:45s} {size_kb:8.1f} KB")


✓ model_multivariate.keras
✓ model_univariate.keras
✓ model_config.json
✓ training_history_multi.json
✓ training_history_uni.json
✓ predictions_test.joblib

SEMUA ARTEFAK MODEL TERSIMPAN:
  best_multi_checkpoint.keras                     2851.7 KB
  best_uni_checkpoint.keras                       1430.0 KB
  grid_results_multivariate.json                    29.8 KB
  grid_results_univariate.json                       8.9 KB
  model_config.json                                  1.8 KB
  model_multivariate.keras                        2851.7 KB
  model_univariate.keras                          1429.7 KB
  predictions_test.joblib                           23.3 KB
  training_history_multi.json                        3.8 KB
  training_history_uni.json                          3.8 KB


## 14. Cara Load Model untuk Forecast / Visualisasi

Cell ini adalah **referensi** — tidak perlu dijalankan ulang di sini,
tapi bisa di-copy langsung ke `evaluasi_visualisasi.ipynb` atau notebook forecast baru.


In [51]:
# ════════════════════════════════════════════════════════════════
# REFERENCE — Load Model untuk Inference / Forecast / Visualisasi
# ════════════════════════════════════════════════════════════════

import os, json, joblib, numpy as np, pandas as pd
import tensorflow as tf
from pathlib import Path

ROOT           = Path(os.getcwd()).parent
MODEL_DIR      = ROOT / "model" / "model_multivariate"
DATA_PROCESSED = ROOT / "data" / "processed"

# ── 1. Load config ──
with open(MODEL_DIR / "model_config.json") as f:
    cfg = json.load(f)

print("Model config loaded:")
print(f"  Multivariate RMSE : {cfg['multivariate']['test_metrics']['rmse']}")
print(f"  Multivariate MAPE : {cfg['multivariate']['test_metrics']['mape']}%")
print(f"  Multivariate R²   : {cfg['multivariate']['test_metrics']['r2']}")
print(f"  Univariate   RMSE : {cfg['univariate']['test_metrics']['rmse']}")
print(f"  Univariate   MAPE : {cfg['univariate']['test_metrics']['mape']}%")
print(f"  Univariate   R²   : {cfg['univariate']['test_metrics']['r2']}")

# ── 2. Load scalers ──
scaler_target  = joblib.load(DATA_PROCESSED / "scaler_target.joblib")
scaler_daily   = joblib.load(DATA_PROCESSED / "scaler_daily.joblib")
scaler_monthly = joblib.load(DATA_PROCESSED / "scaler_monthly.joblib")
scaler_uni     = joblib.load(DATA_PROCESSED / "scaler_univariate.joblib")

# ── 3. Load model ──
model_multi = tf.keras.models.load_model(MODEL_DIR / "model_multivariate.keras")
model_uni   = tf.keras.models.load_model(MODEL_DIR / "model_univariate.keras")
print("\n✓ model_multivariate.keras loaded")
print(f"  Input shapes: {[i.shape for i in model_multi.inputs]}")
print("✓ model_univariate.keras loaded")
print(f"  Input shape : {model_uni.input_shape}")

# ── 4. Load predictions test (sudah tersimpan) ──
preds = joblib.load(MODEL_DIR / "predictions_test.joblib")
y_true_multi = np.array(preds['multivariate']['y_true'])
y_pred_multi = np.array(preds['multivariate']['y_pred'])
dates_multi  = preds['multivariate']['dates']

y_true_uni   = np.array(preds['univariate']['y_true'])
y_pred_uni   = np.array(preds['univariate']['y_pred'])

print(f"\n✓ predictions_test.joblib loaded")
print(f"  Multivariate test samples : {len(y_true_multi)}")
print(f"  Univariate   test samples : {len(y_true_uni)}")

# ── 5. Contoh: 1-step forecast dari data baru ──
# (ini contoh — sesuaikan dengan window terakhir data aktual)
def forecast_next(model_m, model_u, last_daily_window, last_monthly_window,
                  scaler_d, scaler_m, scaler_t, scaler_u):
    """
    Forecast IHSG 1 hari ke depan.
    
    Parameters
    ----------
    last_daily_window   : np.array (seq_daily, 2)  — [IHSG, Kurs], nilai RAW (belum scaled)
    last_monthly_window : np.array (seq_monthly, 2) — [Inflasi, BI Rate], nilai RAW
    
    Returns
    -------
    pred_multi : float — prediksi multivariate (poin IHSG)
    pred_uni   : float — prediksi univariate   (poin IHSG)
    """
    # Scale
    daily_sc   = scaler_d.transform(last_daily_window)           # (seq_d, 2)
    monthly_sc = scaler_m.transform(last_monthly_window)         # (seq_m, 2)
    uni_sc     = scaler_u.transform(last_daily_window[:, 0:1])   # (seq_d, 1)
    
    # Batch dim
    X_d = daily_sc[np.newaxis, ...]    # (1, seq_d, 2)
    X_m = monthly_sc[np.newaxis, ...]  # (1, seq_m, 2)
    X_u = uni_sc[np.newaxis, ...]      # (1, seq_d, 1)
    
    # Predict
    pred_m_sc = model_m.predict([X_d, X_m], verbose=0).flatten()[0]
    pred_u_sc = model_u.predict(X_u, verbose=0).flatten()[0]
    
    # Inverse transform
    pred_multi = scaler_t.inverse_transform([[pred_m_sc]])[0, 0]
    pred_uni   = scaler_u.inverse_transform([[pred_u_sc]])[0, 0]
    
    return pred_multi, pred_uni

print("\n✓ forecast_next() helper defined — siap dipakai di evaluasi_visualisasi.ipynb")
print("   Usage: pred_m, pred_u = forecast_next(model_multi, model_uni,")
print("              last_daily_window, last_monthly_window,")
print("              scaler_daily, scaler_monthly, scaler_target, scaler_uni)")


Model config loaded:
  Multivariate RMSE : 440.097
  Multivariate MAPE : 5.24%
  Multivariate R²   : -1.2198
  Univariate   RMSE : 122.4131
  Univariate   MAPE : 1.3289%
  Univariate   R²   : 0.8283

✓ model_multivariate.keras loaded
  Input shapes: [(None, 90, 2), (None, 12, 2)]
✓ model_univariate.keras loaded
  Input shape : (None, 90, 1)

✓ predictions_test.joblib loaded
  Multivariate test samples : 485
  Univariate   test samples : 485

✓ forecast_next() helper defined — siap dipakai di evaluasi_visualisasi.ipynb
   Usage: pred_m, pred_u = forecast_next(model_multi, model_uni,
              last_daily_window, last_monthly_window,
              scaler_daily, scaler_monthly, scaler_target, scaler_uni)


## Summary Modeling

| Tahap | Detail |
|---|---|
| **Grid Search Multi** | seq_daily × seq_monthly × lstm_units × dropout × batch × lr |
| **Grid Search Uni** | seq_daily × lstm_units × dropout × batch × lr |
| **Final Training** | EarlyStopping + ReduceLROnPlateau, restore best weights |
| **Evaluasi** | RMSE, MAPE, R² pada test set (skala asli IHSG) |
| **Model disimpan** | `.keras` native format — load dengan `tf.keras.models.load_model()` |
| **Config disimpan** | `model_config.json` — berisi semua hyperparameter + metrik |

---

### Next Step: `evaluasi_visualisasi.ipynb`
Load `model_config.json` + `predictions_test.joblib` + kedua model `.keras`, lalu:
1. Plot prediksi vs aktual (Plotly interaktif, hover-enabled)
2. Tabel perbandingan metrik Univariate vs Multivariate
3. Plot training history (loss curve kedua model)
4. Residual analysis
5. Optional: rolling forecast jika diperlukan
